# 第14章 トークナイザーを自作する：BPE

よく出るバイトの組を、くり返し合体させるだけ

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：バイトレベルBPEトークナイザー（学習・エンコード・デコード）をゼロから実装し、Hugging Faceの `tokenizers` で作ったものと比べる
- **身につく数学・理論**：BPE（バイトペア符号化）のアルゴリズム ／ マージ規則の順番と、エンコードの手順 ／ 語彙サイズと系列長のトレードオフ ／ トークナイザーが違うモデルの比べ方（1文字あたりの損失） ／ Unicode正規化（NFKC）と日本語の注意点
- **新しく使うPython・PyTorch**：`bytes` と `list(…)` で数の列に ／ 辞書の `get(キー, 初期値)` ／ `max(…, key=…)` ／ `bytes.decode(errors="replace")` ／ `tokenizers` ライブラリ

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 2時間 |
| 難易度 | ★★★★☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> 第2章で、Qwenのトークナイザーが「よく出るバイトの並びを1トークンにまとめている」ことを見ました。この章では、その仕組み **BPE**（Byte Pair Encoding）を自分で実装します。アルゴリズムは驚くほど単純で、「いちばんよく出る隣どうしのペアを合体させる」を何百回もくり返すだけ。最後に、Hugging Faceの高速な実装と比べて、自作GPTをBPEで学習させてみます。

> **🔗 これまでの復習とつながり：第2章とのつながり**
>
> - **第2章**：テキストはUTF-8の **バイト列（0〜255の整数）** で表せ、日本語1文字はおおよそ3バイトになることを見ました。また、QwenのバイトレベルBPEでは「鰯」のような珍しい漢字が3バイトの断片に分かれても未知語にならず完全に復元できました。
> - **第12〜13章**：これまでの自作GPTは「1文字＝1トークン」で動かしてきましたが、文章が長くなりやすいという限界がありました。
>
> この章では、その2つをつなぎ、0〜255のバイト列から出発して「よく出るペアを合体させる」**バイトレベルBPE**をゼロから自作します。

**ミナト**：第2章で見た「鰯」が3バイトのまま残っていた理由が、ついに分かるんですね。

**ボンゴ先生**：そう。BPEは、もともと1994年に「データ圧縮」の方法として考えられた。ZIPファイルの親戚だね。それが20年以上たって、LLMの入り口になった。研究の世界では、昔の小さなアイデアが、思わぬところで主役になることがよくある。キミの基本情報の勉強も、いつか思わぬところで主役になるかもしれないよ。

## まずは手で体験

下のデモで、「1回合体」を何回か押してみてください。BPEのすべてがここにあります。

*(対話デモはWeb教科書ページで確認できます：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html)*

1. 隣り合う2つのトークンの組を全部数える
2. いちばん多い組を、新しい1つのトークンに合体させる（この「合体ルール」を覚えておく）
3. 1に戻る

これだけです。合体のたびにトークン数が減り（文章が短くなり）、語彙の種類が増えます。

#### 📝 確認問題（`ch14-q01` · Lv.1 · BPEの学習ループと1回の合体で起きる変化）

BPE（Byte Pair Encoding）の学習で、最も出現回数が多い隣り合うペアを1回合体（マージ）すると、文章のトークン列の長さと語彙の種類数はそれぞれどう変わりますか？

- **a.** トークン列は長くなり、語彙の種類数は1つ減る
- **b.** トークン列の長さも語彙の種類数も変化せず一定のまま保たれる
- **c.** トークン列は短くなり、語彙の種類数は1つ増える
- **d.** トークン列は短くなり、元の要素が消えるため語彙の種類数も減る

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：2個並んでいたトークンを、新しい1個のトークンIDに置き換える操作を思い浮かべてみましょう。
- **ヒント 2**：合体のたびに元の要素は辞書から消さず、新IDを1件追加します。列の長さと語彙数の増減をそれぞれ整理しましょう。
- **正解**：`c`

**考え方**：BPEは「出現頻度が最大のペアを新しい1トークンに置き換える」操作を繰り返すことで、語彙を増やす代わりに文章のトークン列を圧縮します。

1. 最頻ペア `(a, b)` が文章中に $c$ 回出現しているとする。
1. それをすべて新しいID `new_id`（1個）に置き換えると、列の長さは $c$ だけ短くなる。
1. 辞書に `new_id` が1種類追加されるため、語彙の種類数は `+1` される。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html#ex-ch14-q01

</details>

## 準備：文章をバイトにする

> **✨ コツ：実行環境とリソースの目安**
>
> - **環境**：CPUで完走できます（GPU不要）。
> - **ダウンロードとメモリ**：初回に青空文庫『こころ』（約350KB）と教材コード `minigpt.py`（約4KB）を取得します。必要RAMは約1GBです。
> - **所要時間**：自作BPEの300マージはCPUで1〜2分、`tokenizers` 版は数秒、章末のBPE版ミニGPT学習（1500ステップ）はCPUで3〜6分（GPUなら十数秒）です。時間を短縮したいときは `range(1501)` を `range(501)` にしても傾向を確認できます。
> - **ライブラリ**：推奨バージョンはリポジトリの `requirements-book.txt` を参照してください。

本物のBPEは、文字ではなく **UTF-8のバイト**（0〜255の数）から始めます（第2章）。こうすると、どんな文字でも最初から表せるので、「辞書にない文字」が存在しなくなります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import re, urllib.request

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/"
           "0984f7dc18a2270bdfd2ff99bf14fd1b3ebb4f65/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    try:
        text = urllib.request.urlopen(url, timeout=30).read().decode("cp932")
    except Exception as e:
        raise RuntimeError(
            f"青空文庫の取得に失敗しました（{e}）。ネットワーク接続を確認して再実行するか、付録E（エラー辞典）を参照してください。"
        ) from e
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
train_text = text[:30000]               # 自作版は遅いので、最初の3万字で学習
ids = list(train_text.encode("utf-8"))
print("文字数：", len(train_text), " バイト数：", len(ids))
print("最初の12バイト：", ids[:12])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
文字数： 30000  バイト数： 89160
最初の12バイト： [228, 184, 138, 227, 128, 128, 229, 133, 136, 231, 148, 159]
```

</details>

3万字が約9万バイト。日本語は1文字がだいたい3バイトでしたね。`228, 184, 138` で「上」、`227, 128, 128` で全角空白です。

#### 📝 確認問題（`ch14-q02` · Lv.2 · UTF-8バイト（0〜255）から始める理由）

現代のLLM（GPT-2やQwenなど）のBPEが、文字ではなく **UTF-8のバイト（0〜255の整数）** を出発点にしている最大の理由は何ですか？

- **a.** 基礎単位を256種類に固定することで、合体後の語彙サイズも常に256以下に抑えられるから
- **b.** 入力を0〜255のバイト列に変換しておくと、Transformer内部のAttention計算が不要になるから
- **c.** あらゆる文字や絵文字を256種のバイト列で表せるため、未知語（表現不能な文字）がゼロになるから
- **d.** 日本語のひらがなや漢字がすべて1バイトで表現でき、英語よりトークン列を短く圧縮できるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：学習データに1度も登場しなかった珍しい漢字や新しい絵文字が、テスト時に現れたらどうなるか考えてみましょう。
- **ヒント 2**：文字単位の辞書では未知文字が `<UNK>` になりますが、UTF-8の1バイト（0〜255の256種類）をすべて初期語彙に入れておくと何が保証されるでしょうか？
- **正解**：`c`

**考え方**：0〜255の256個のバイトを基礎語彙として必ず持っておくことで、あらゆるUTF-8文字列を未知語（`<UNK>`）なしで可逆的に符号化できます。

1. あらゆる文字・記号・絵文字はUTF-8で0〜255のバイト列に変換できる。
1. 初期語彙に0〜255の256個をすべて入れておけば、未見の文字が来ても最小1バイト単位まで分解して表現できる。
1. その結果、未知語（OOV）が原理的にゼロになる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html#ex-ch14-q02

</details>

## ステップ1：ペアを数える

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(ids)
top = sorted(stats.items(), key=lambda kv: -kv[1])[:5]
for pair, count in top:
    print(pair, count, "回 →", repr(bytes(pair).decode("utf-8", errors="replace")))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
(227, 129) 15275 回 → '�'
(227, 130) 3720 回 → '�'
(227, 128) 2119 回 → '�'
(159, 227) 1456 回 → '��'
(139, 227) 1339 回 → '��'
```

</details>

> **🐍 Python・PyTorchのポイント：get と lambda**
>
> `counts.get(pair, 0)` は「辞書に `pair` があればその値、なければ0」を返します。初めて見たペアでもエラーにならずに数え始められる、便利な書き方です。<br>
> `sorted(…, key=lambda kv: -kv[1])` は「各要素 `kv` の2番目（回数）にマイナスを付けた値で並べ替える」、つまり回数の多い順です。`lambda` は、その場で使い捨ての小さな関数を作る書き方です。<br>
> 次の `merge` 関数で使う `while 条件:` は、「条件が正しいあいだ、くり返す」ループです。`for` と違って、何回くり返すかを先に決めずに、位置 `i` を自分で1つ進めたり2つ進めたりしたいときに使います。フローチャートの「条件つきの戻り矢印」そのものです。

一番多いのは `(227, 129)` で、1万5千回以上。これはひらがなの多く（「ぁ」〜「み」あたり）に共通する、最初の2バイトです。ひらがなは日本語で一番よく出てくる文字なので、そのバイトの組も一番多いわけです。バイトの切れ端だけでは文字にならないので `�` と表示されています。4位と5位の `(159, 227)` や `(139, 227)` は、「ある文字の最後のバイト」と「次の文字の最初のバイト」の組です。BPEは文字の切れ目を知らないので、こういう組も数えます。

<details><summary><b>基礎の補足：理解のための最小トレース：`zip(ids, ids[1:])` で隣接ペアを数える</b>（クリックで開く）</summary>

長さ5の小さな列 `ids = [1, 2, 1, 2, 3]` で `get_stats(ids)` がどう動くか、1歩ずつ追いかけましょう。

```python no-run
ids = [1, 2, 1, 2, 3]
# ids     :  1,  2,  1,  2,  3
# ids[1:] :  2,  1,  2,  3
# zip結果 : (1,2), (2,1), (1,2), (2,3)   ← 長さ5から4個のペアができる
```

| ステップ | `pair` | `counts.get(pair, 0)` | 更新後の `counts[pair]` |
|---|---|---|---|
| 1 | `(1, 2)` | 0 | **1** |
| 2 | `(2, 1)` | 0 | **1** |
| 3 | `(1, 2)` | 1 | **2** ← 最頻ペア！ |
| 4 | `(2, 3)` | 0 | **1** |

元のリストが長さ $n$ のとき、隣接ペアは必ず **$n - 1$ 個**になります（長さ0や1なら0個で空辞書 `{}`）。

</details>

#### 📝 確認問題（`ch14-q03` · Lv.2 · zip(ids, ids[1:]) による隣接ペア集計の追跡）

次のコードを実行すると、何が表示されますか？

```python
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats([1, 2, 1, 2, 3, 1, 2])
print(stats[(1, 2)], stats[(2, 1)])
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`zip(ids, ids[1:])` は、リストの隣り合う2要素を左から1つずつずらしながらペアとして取り出します。
- **ヒント 2**：`ids` の長さは7なので隣接ペアは全部で6個できます。最初の2個は `(1, 2)` と `(2, 1)` です。残りの4個も書き出してそれぞれの出現回数を数えましょう。
- **期待される出力**：`3 1`

**考え方**：`zip(ids, ids[1:])` は長さ $n$ の列から $n - 1$ 個の隣接ペアを順番に取り出します。

1. `ids = [1, 2, 1, 2, 3, 1, 2]`（長さ7）なので隣接ペアは6個。
1. ペア列：`(1,2)`, `(2,1)`, `(1,2)`, `(2,3)`, `(3,1)`, `(1,2)`。
1. `(1, 2)` は3回、`(2, 1)` は1回 → `3 1`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html#ex-ch14-q03

</details>

## ステップ2：合体させる

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def merge(ids, pair, new_id):
    out = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id)     # ペアを見つけたら、新しいIDに置き換えて2つ進む
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

print(merge([5, 6, 6, 7, 9, 6, 7], (6, 7), 99))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
[5, 6, 99, 9, 99]
```

</details>

`(6, 7)` が2か所とも `99` に置き換わりました。

<details><summary><b>基礎の補足：理解のための最小トレース：`merge(ids, pair, new_id)` の動きと「連続重複」の落とし穴</b>（クリックで開く）</summary>

`merge` が `for` ではなく `while` を使う理由は、ペアが見つかったときに **2歩進む（`i += 2`）**、見つからないときは **1歩進む（`i += 1`）** と歩幅を切り替えたいからです。

特に注意したいのが、同じIDが3回連続する `[5, 5, 5]` に `(5, 5) -> 99` を適用するケースです。

| 位置 `i` | 見ている値 | `(5, 5)` と一致？ | `out` に追加 | 次の `i` |
|---|---|---|---|---|
| `i = 0` | `ids[0]=5, ids[1]=5` | はい | `99` | `0 + 2 = 2` |
| `i = 2` | `ids[2]=5`（末尾なので `i < len(ids) - 1` は偽） | いいえ | `5` | `2 + 1 = 3`（終了） |

結果は **`[99, 5]`** になります。左から貪欲に2個ずつ消費していくため、3個目の `5` はそのまま残ります。また、`if i < len(ids) - 1 and ...` を左側に書くことで、末尾要素のときに `ids[i + 1]` を読んで `IndexError` になるのを防いでいます。

</details>

#### 📝 確認問題（`ch14-q04` · Lv.2 · merge() のポインタ移動と連続重複ペアの扱い）

同じ数字が3回連続する `[5, 5, 5, 8]` に対して、ペア `(5, 5)` を `99` に合体します。出力を予測してください。

```python
def merge(ids, pair, new_id):
    out = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out

print(merge([5, 5, 5, 8], (5, 5), 99))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`merge` は左から順に走査し、指定ペアと一致したときにインデックス `i` をどれだけ進めるかがポイントです。
- **ヒント 2**：`i = 0` で `(5, 5)` が一致すると `out` に `99` が入り、`i += 2` で `i = 2` にジャンプします。残る `ids[2]` 以降がどう処理されるか追いましょう。
- **期待される出力**：`[99, 5, 8]`

**考え方**：`merge` は左から順に走査し、一致したら `i += 2` で2つ消費するため、重なり合う3連の `[5, 5, 5]` は左の2個だけが `99` になり、3個目の `5` はそのまま残ります。

1. `i=0`：`ids[0]==5, ids[1]==5` → `out=[99]`、`i` を `2` に進める。
1. `i=2`：`ids[2]==5, ids[3]==8` → 不一致なので `out=[99, 5]`、`i=3`。
1. `i=3`：末尾なので `out=[99, 5, 8]`、`i=4` で終了。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html#ex-ch14-q04

</details>

#### 📝 確認問題（`ch14-q05` · Lv.4 · 【自力実装】隣接ペアの出現回数を数える get_pair_counts）

整数リスト `ids` を受け取り、隣り合う2要素のタプル `(a, b)` をキー、出現回数を値とする辞書を返す関数 `get_pair_counts(ids)` を標準Pythonだけで自力実装し、検算テストを通してください（空リストや長さ1のリストでは空辞書 `{}` を返すこと）。

**ひな形コード：**
```python
def get_pair_counts(ids):
    # counts = {} を用意し、zip(ids, ids[1:]) で隣接ペアを回して
    # counts[pair] = counts.get(pair, 0) + 1 で数える
    pass
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：空の辞書を用意し、リストから隣り合う2要素のタプルを順に取り出してカウントを1ずつ増やします。
- **ヒント 2**：`for pair in zip(ids, ids[1:]):` を使うと、長さ0や1のリストでも1回も回らず自然に `{}` を返せます。`counts.get(pair, 0) + 1` で加算しましょう。

**解答例：**
```python
def get_pair_counts(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts
```

**検算テスト（Colabで実行して確認）：**
```python
assert get_pair_counts([]) == {}
assert get_pair_counts([42]) == {}
assert get_pair_counts([1, 2, 1, 2, 3]) == {(1, 2): 2, (2, 1): 1, (2, 3): 1}
assert get_pair_counts([5, 5, 5]) == {(5, 5): 2}
print("ALL PASSED")
```

**考え方**：BPEの学習とエンコードの両方で毎回呼ばれる基本部品です。`zip(ids, ids[1:])` と `dict.get(pair, 0)` の組み合わせはPythonらしい簡潔なイディオムです。

1. 空の辞書 `counts = {}` を作成する。
1. `for pair in zip(ids, ids[1:]):` で隣接2要素をタプルとして取り出す。
1. `counts[pair] = counts.get(pair, 0) + 1` で回数を加算し、`counts` を返す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html#ex-ch14-q05

</details>

*(この節の追加問題 `ch14-q06` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html で自動採点・復習できます)*

## ステップ3：くり返して学習する

0〜255はバイトそのものに使っているので、新しいトークンは256番から番号をつけていきます。300回合体させて、語彙556個のトークナイザーを作りましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
num_merges = 300
merges = {}                     # (a, b) → 新しいID（合体ルール）
vocab = {i: bytes([i]) for i in range(256)}   # ID → バイト列
work = list(ids)
for k in range(num_merges):
    stats = get_stats(work)
    pair = max(stats, key=stats.get)            # いちばん多いペア
    new_id = 256 + k
    work = merge(work, pair, new_id)
    merges[pair] = new_id
    vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]

print("学習前：", len(ids), "トークン → 学習後：", len(work), "トークン")
print("圧縮率：", round(len(ids) / len(work), 2), "倍")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
学習前： 89160 トークン → 学習後： 28975 トークン
圧縮率： 3.08 倍
```

</details>

`max(stats, key=stats.get)` は「回数が最大のペア」を選ぶ書き方です。CPUで1〜2分かかります。300回の合体で、約9万バイトが約2万9千トークンに、3分の1ほどに縮みました。1トークンに平均3バイト、つまり日本語なら平均1文字ほどです。どんなトークンができたか、のぞいてみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
learned = [vocab[256 + k].decode("utf-8", errors="replace") for k in range(num_merges)]
print("最初の30個：", learned[:30])
print("最後の30個：", learned[-30:])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
最初の30個： ['�', '�', '�', 'の', 'た', 'い', '。', 'に', 'っ', 'は', 'な', 'て', 'し', 'か', 'で', 'た。', 'と', 'を', '、', 'る', 'う', '�', '私', 'が', 'ら', 'も', 'そ', 'れ', '�', '�']
最後の30個： ['」とい', '�', '会', '意', 'か」\n「', 'より', '知', '�', '私には', '始', 'そこ', '��', '二人', '向', 'ありません', '掛', '来た', '変', '好', '手', 'の上', 'るの', '付', '��', '外', '自分', '�', '��え', '��', '然']
```

</details>

最初のうちは、バイトの組み合わせから「の」「た」「い」などの**ひらがな1文字**や、「私」のようなよく出る漢字が作られていきます。後半になると、「ありません」「私には」「自分」「二人」「来た」のような**よく出る言葉や言い回し**がまるごと1トークンになっています。「」とい」「か」\n「」のように、会話のかぎかっこの切れ目をまたいだトークンもできていますね。誰も日本語の文法を教えていないのに、「よく出る組を合体させる」だけで、言葉らしい単位が浮かび上がってきました。

ところどころに残っている `�` は、文字の**途中までのバイト**を合体させたトークンです。たとえば、ある漢字の最初の2バイトは、その漢字の仲間（同じ部首の漢字など）に共通することがあるので、それだけで合体してしまうのです。

#### 📝 確認問題（`ch14-q07` · Lv.2 · 初期バイト256種＋マージ回数＝語彙サイズ）

UTF-8の1バイト（ID 0〜255の256種類）から始めて、BPEの合体を **300回** 行いました。最終的に語彙サイズはいくつになりますか？（整数で答えてください）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：バイトレベルBPEでは、初期語彙（1バイトの全種類）に、合体を行うたびに新トークンIDが1つずつ追加されます。
- **ヒント 2**：初期語彙はID 0〜255の256個です。そこに300回の合体で追加される個数を足し合わせましょう。
- **正解**：`556` 語

**考え方**：バイトレベルBPEでは0〜255の256個がベースとなり、$k$ 回の合体でID `256`〜`256 + k - 1` が追加されるため、総語彙数は $256 + k$ になります。

1. 初期語彙：0〜255の256個。
1. 300回の合体でID 256〜555の300個を追加。
1. $256 + 300 = 556$。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html#ex-ch14-q07

</details>

## エンコードとデコード

学習した合体ルールを使って、新しい文章をトークンに変える（エンコード）関数と、元に戻す（デコード）関数を作ります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def decode(token_ids):
    return b"".join(vocab[t] for t in token_ids).decode("utf-8", errors="replace")

def encode(s):
    ids = list(s.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        # 学習で「先に」覚えたルール（番号が小さいもの）から順に使う
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break                       # もう使えるルールがない
        ids = merge(ids, pair, merges[pair])
    return ids

sample = "先生は奥さんと話をしていた。"
tokens = encode(sample)
print(len(sample.encode("utf-8")), "バイト →", len(tokens), "トークン")
print([decode([t]) for t in tokens])
print(decode(tokens) == sample)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
42 バイト → 8 トークン
['先生は', '奥さん', 'と', '�', '��', 'を', 'して', 'いた。']
True
```

</details>

42バイトが8トークンになりました。「先生は」「奥さん」「いた。」がまるごと1トークンです。一方、「話」は学習に使った最初の3万字にあまり出てこなかったので、`�` と `��` の2つのバイトの切れ端に分かれています（3バイトが1＋2に分かれた）。それでも最後の行の `True` のとおり、つなげて読めば元の文章に完全に戻ります。第2章の「鰯」と同じことが、自分のトークナイザーで起きたわけです。

> **💡 理論のポイント：エンコードは「学習した順番」でルールを使う**
>
> エンコードでは、文章をバイトに分解したあと、**学習したときと同じ順番**で合体ルールを適用します。「先に覚えたルール（番号が小さい）」が、学習データでよく出てきた組だからです。順番を変えると、学習時とは違う区切り方になってしまいます。
>
> デコードはもっと簡単で、各トークンのバイト列をつなげて、UTF-8として読むだけです。`errors="replace"` は「途中で切れたバイトがあっても、エラーにせず `�` にする」指定です。LLMが生成の途中でバイトの途中までしか出していない瞬間があるので、これが必要になります。

<details><summary><b>基礎の補足：なぜ「学習した順番（最小ID優先）」が必須なのか？具体例で確かめる</b>（クリックで開く）</summary>

学習で次の順番に3つのルールを覚えたとします（番号が小さいほど先に学習したルール）。

1. `(1, 2) → 256`（1番目に学習）
2. `(256, 3) → 257`（2番目に学習：`256` が作られて初めて使える！）
3. `(2, 3) → 258`（3番目に学習）

ここで入力 `[1, 2, 3]` をエンコードしてみましょう。

```python no-run
# 正しい手順：学習順（番号が小さいルールから）適用する
# [1, 2, 3] → (1,2)が256、(2,3)が258 → 小さい256を先に適用 → [256, 3]
# [256, 3]  → (256,3)が257 → 適用 → [257]   ← 学習時と同じ1トークン！

# 誤った手順：順番を無視して (2, 3) → 258 を先に適用してしまう
# [1, 2, 3] → [1, 258]   ← もう (256, 3) を作れない！ 学習時と違う2トークンに分裂！
```

だから `encode` では、`min(stats, key=lambda p: merges.get(p, float("inf")))` と書いて、**いま並んでいるペアのうち、ルール番号がもっとも小さいもの（＝いちばん先に学習したルール）**を毎回1つ選んで合体します。ルールに載っていないペアには `float("inf")`（無限大）を返すので、使えるルールが1つでもあれば必ずそちらが選ばれます。

</details>

#### 📝 確認問題（`ch14-q08` · Lv.3 · エンコード時に合体ルールを「学習した順番」で適用する理由）

BPEのエンコード（`encode(s)`）で、文章中のペアに対して合体ルールを適用するとき、なぜ「**学習したときと同じ順番（新しく割り当てたIDが小さい順）**」で適用しなければならないのですか？

- **a.** 順番を変えても最終的なトークン列は必ず一致するが、IDが小さい順のほうが探索速度が少し速いから
- **b.** 後のルールは先のルールで作ったトークンを部品にしており、順を変えると学習時と別物の列になるから
- **c.** IDが大きいルールから先に参照すると、Pythonの辞書探索が順序違反の例外を出して停止するから
- **d.** 学習時の順番で合体しないと、各トークンIDのバイト列が壊れて `decode` で元文字列へ戻せなくなるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：BPEのトークンは、先の合体で作られたIDを部品にして次の合体が行われる階層構造を持っています。
- **ヒント 2**：`(a, b) -> 256`、次に `(256, c) -> 257`、さらに後で `(b, c) -> 258` を学習したとき、`[a, b, c]` で `(b, c)` を先に使うとどうなるか考えてみましょう。
- **正解**：`b`

**考え方**：BPEのトークンは木構造のように段階的に組み立てられています。学習時と同じ優先順位（`merges[pair]` が最小のペア）を再現しないと、下位の合体が飛ばされて別物のトークン列になってしまいます。

1. 学習時は `(a, b) -> 256` が先、次に `(256, c) -> 257`、最後に `(b, c) -> 258` と覚えたとする。
1. `[a, b, c]` を学習順でエンコードすると `[a, b, c] → [256, c] → [257]`。
1. もし順番を無視して `(b, c)` を先に使うと `[a, 258]` になり、学習時と違うトークン列になってしまう。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html#ex-ch14-q08

</details>

#### 📝 確認問題（`ch14-q09` · Lv.3 · 具体的なマージ規則でのエンコード結果の追跡）

次のミニBPEエンコーダーを実行すると、最終的なトークンID列は何になりますか？

```python
def get_stats(ids):
    c = {}
    for p in zip(ids, ids[1:]):
        c[p] = c.get(p, 0) + 1
    return c

def merge(ids, pair, new_id):
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

merges = {(1, 2): 256, (256, 3): 257, (2, 3): 258}
ids = [1, 2, 3, 2, 3]
while len(ids) >= 2:
    stats = get_stats(ids)
    pair = min(stats, key=lambda p: merges.get(p, float("inf")))
    if pair not in merges:
        break
    ids = merge(ids, pair, merges[pair])
print(ids)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`while` ループでは毎回、現在の列に存在するペアのうち `merges` のID番号が最も小さいルールを1つ選んで合体します。
- **ヒント 2**：`[1, 2, 3, 2, 3]` では `(1, 2): 256` と `(2, 3): 258` が使えます。小さい `256` を先に適用して `[256, 3, 2, 3]` とし、続く合体を最後まで追いましょう。
- **期待される出力**：`[257, 258]`

**考え方**：番号の小さい順に `(1, 2) -> 256` → `(256, 3) -> 257` → `(2, 3) -> 258` と3段階で合体が進み、`[257, 258]` になります。

1. 初期：`[1, 2, 3, 2, 3]`。使えるルールは `(1,2):256` と `(2,3):258` → 最小の `256` を適用 → `[256, 3, 2, 3]`。
1. 次：使えるルールは `(256,3):257` と `(2,3):258` → 最小の `257` を適用 → `[257, 2, 3]`。
1. 次：使えるルールは `(2,3):258` → 適用 → `[257, 258]`。
1. `(257, 258)` はルールにないので終了。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html#ex-ch14-q09

</details>

*(この節の追加問題 `ch14-q10` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html で自動採点・復習できます)*

## 本格版：Hugging Faceの tokenizers

自作版は仕組みを学ぶのに最適ですが、遅すぎて本物のデータには使えません。実務では、高速に書かれたライブラリを使います。Hugging Faceの `tokenizers` は、内部がRustという言語で書かれていて、自作版の何百倍も速く動きます。『こころ』全体で、語彙4000のバイトレベルBPEを学習させてみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

bpe = Tokenizer(models.BPE())
bpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=4000, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
bpe.train_from_iterator(text.split("\n"), trainer=trainer)

enc = bpe.encode("先生は奥さんと話をしていた。")
print(enc.ids)
print([bpe.decode([i]) for i in enc.ids])

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
[485, 1636, 1891, 685, 264]
['先生は', '奥さんと', '話を', 'していた', '。']
```

</details>

一瞬で終わりました。『こころ』全体で学習し、語彙も大きいので、「奥さんと」「話を」「していた」がそれぞれ1トークン。文全体がたった5トークンです。

> **💡 理論のポイント：語彙サイズのトレードオフ（サブワードの絶妙なバランス）**
>
> BPEのような**サブワード**（単語の断片）トークナイザーは、「文字単位」と「単語単位」のいいとこ取りをした架け橋です。
>
> - **文字単位（第13章まで）**：語彙数は数千文字でコンパクトですが、文章のトークン列がとても長くなります。Attentionの計算量は列の長さの2乗（$T \times T$）で増えるため、文章が長いと計算が急激に重くなります。
> - **単語単位**：文章のトークン列は短くなりますが、日本語や英語のすべての単語を網羅しようとすると語彙数が数十万〜数百万に爆発し、辞書にない未知語（新語や誤字）を扱えなくなります。
> - **サブワード（BPE）**：よく出る単語や言い回しは1トークンにまとめ、珍しい単語は細かい断片（最後は1バイト単位）に分解して表します。未知語を一切出さずに、列の長さを適度に短く抑えられます。
>
> 語彙サイズ（合体回数）をどれくらいにするかはトレードオフです。
>
> - 👍 語彙を大きくすると、同じ文章が**少ないトークン**で表せる（列が短くなり、計算が軽く、遠くまで見渡せる）
> - 👎 語彙を大きくしすぎると、埋め込みと出力層のパラメータ（語彙数 $\times$ 幅）が膨らみ、めったに出てこないトークンの学習が追いつかなくなる
>
> 実際のLLMの語彙サイズは、GPT-2が約5万、Qwen2.5が約15万、多言語のモデルほど大きい傾向です。日本語・中国語・英語……と多くの言語の「よく出る組」を入れるには、大きな辞書が必要だからです。

<details><summary><b>基礎の補足：数字で見る語彙サイズのトレードオフ（$V \times d$ vs $T^2$）</b>（クリックで開く）</summary>

埋め込み幅 $d = 768$ のモデルで、語彙サイズ $V$ を変えたときの重みと計算量のバランスを比べてみましょう。

| 語彙サイズ $V$ | 埋め込み表 `wte` のパラメータ数（$V \times 768$） | 同じ日本語文章のトークン数 $T$（目安） | Attentionのスコア表サイズ（$T^2$） |
|---|---|---|---|
| 256（バイトそのまま） | 約20万個（極小） | 約3,000（長い） | 900万マス（重い） |
| 2,000（小型BPE） | 約154万個 | 約650 | 42万マス |
| 50,000（GPT-2級） | 約3,840万個 | 約400 | 16万マス |
| 150,000（多言語LLM級） | 約1.15億個（巨大） | 約300（短い） | 9万マス（軽い） |

語彙を増やすほど $T$ が短くなって $T^2$ のAttentionが激減する一方、埋め込み層と出力層のサイズ $V \times d$ が直線的に膨らみます。そのため、モデルの大きさや対応言語数に合わせてちょうど良い $V$ を選びます。

</details>

#### 📝 確認問題（`ch14-q11` · Lv.3 · 語彙サイズのトレードオフ（系列長 vs パラメータ数・学習密度））

LLMの語彙サイズ $V$ を大きくしたときの**メリット**と**デメリット**の組み合わせとして正しいものはどれですか？

- **a.** 長所：埋め込み層と出力層のパラメータ数が大幅に減る ／ 短所：同じ文章のトークン数 $T$ が長くなり処理が重くなる
- **b.** 長所：Attentionの計算量が語彙サイズ $V$ に比例して小さくなる ／ 短所：推論時の生成ステップが速くなりすぎる
- **c.** 長所：学習データが少なくても過学習が絶対に起きなくなる ／ 短所：語彙が増えると英語以外の言語を表現できなくなる
- **d.** 長所：文章のトークン数 $T$ が減り計算が軽くなる ／ 短所：埋め込み・出力層の $V \times d$ が肥大化し希少語が学習不足になる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：語彙サイズ $V$ を増やすと、文章のトークン列の長さ $T$ と、モデルの入出力行列のサイズ $V \times d$ がそれぞれどう変化するか考えましょう。
- **ヒント 2**：長い言い回しが1トークンにまとまると $T$ は縮みますが、一方で埋め込み表・出力層の行数と各トークンの出現頻度にどんな影響が出るか整理しましょう。
- **正解**：`d`

**考え方**：語彙サイズ設計は「系列長 $T$ の短縮（Attentionと生成ステップの高速化）」と「埋め込み・出力層の肥大化・希少トークンのデータ不足」のバランス取りです。

1. 語彙 $V$ を大きくする → よく出る長い言い回しが1トークンになり、列の長さ $T$ が短くなる。
1. 一方で、埋め込み行列 `wte` と出力層 `lm_head` のサイズは $V \times d$ に比例して増大する。
1. さらに1語あたりの出現回数が分散し、めったに出ないトークンの埋め込みが十分に学習されにくくなる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html#ex-ch14-q11

</details>

## BPEで自作GPTを学習させる

最後に、第12章のGPTを、文字ではなくBPEのトークンで学習させてみましょう。語彙は少し小さめの2000にします。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
!wget -nv https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/minigpt.py
import os, math
import torch
if not os.path.exists("minigpt.py"):
    raise RuntimeError("minigpt.py のダウンロードに失敗しました。ネットワーク接続を確認して再実行するか、付録E（エラー辞典）を参照してください。")
from minigpt import GPT

trainer = trainers.BpeTrainer(vocab_size=2000, initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
bpe.train_from_iterator(text.split("\n"), trainer=trainer)

n = int(len(text) * 0.9)
train_ids = torch.tensor(bpe.encode(text[:n]).ids)
val_ids = torch.tensor(bpe.encode(text[n:]).ids)
print("検証データ：", len(text[n:]), "文字 →", len(val_ids), "トークン")
print("1トークンあたり平均", round(len(text[n:]) / len(val_ids), 2), "文字")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
検証データ： 16247 文字 → 10531 トークン
1トークンあたり平均 1.54 文字
```

</details>

1トークンに平均1.5文字ほど詰め込めています。同じ窓の大きさ（64トークン）でも、文字で数えると約1.5倍遠くまで見渡せることになります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def get_batch(data, batch_size, T, g):
    ix = torch.randint(0, len(data) - T, (batch_size,), generator=g)
    return (torch.stack([data[i:i + T] for i in ix]),
            torch.stack([data[i + 1:i + T + 1] for i in ix]))

@torch.no_grad()
def val_loss_per_token(model, data, T):
    model.eval()
    starts = range(0, len(data) - T - 1, T)
    xs = torch.stack([data[i:i + T] for i in starts])
    ys = torch.stack([data[i + 1:i + T + 1] for i in starts])
    loss = model(xs, ys)[1].item()
    model.train()
    return loss

torch.manual_seed(0)
g = torch.Generator().manual_seed(0)
model = GPT(bpe.get_vocab_size(), block_size=64, n_embd=64, n_head=4, n_layer=2, dropout=0.1)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(1501):
    x, y = get_batch(train_ids, 32, 64, g)
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        vl = val_loss_per_token(model, val_ids, 64)
        print(f"ステップ {step:4d}：検証（1トークンあたり）{vl:.3f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
ステップ    0：検証（1トークンあたり）7.533
ステップ  500：検証（1トークンあたり）4.975
ステップ 1000：検証（1トークンあたり）4.930
ステップ 1500：検証（1トークンあたり）4.976
```

</details>

**ミナト**：4.98……？ 第12章の文字版は3.10だったのに、ずっと悪くなってませんか？

**ボンゴ先生**：ここが大事な落とし穴だ！ 文字版の損失は「**1文字**あたりの驚き」、BPE版は「**1トークン**あたりの驚き」。1トークンには平均1.5文字入っているから、1トークンを当てるのは、1.5文字ぶんをまとめて当てるのと同じくらい難しい。単位が違うものを比べてはいけない。メートルとインチを比べるようなものだ。

> **💡 理論のポイント：1文字あたりの損失に直して比べる**
>
> トークナイザーが違うモデルを比べるときは、**文章全体の驚きの合計**を、**文字数**（またはバイト数）で割って比べます。
>
> $$\text{1文字あたりの損失} = \frac{\text{1トークンあたりの損失} \times \text{トークン数}}{\text{文字数}}$$
>
> **小さな数字で手計算**：たとえば検証データが **10,000文字** で、BPEでは **4,000トークン**（1トークンあたり平均2.5文字）になったとします。このとき1トークンあたりの損失が `3.0` なら、文章全体の総損失は $3.0 \times 4{,}000 = 12{,}000$。これを文字数 $10{,}000$ で割ると、1文字あたり $12{,}000 \div 10{,}000 = 1.2$ になります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
per_token = val_loss_per_token(model, val_ids, 64)
per_char = per_token * len(val_ids) / len(text[n:])
print(f"1トークンあたり {per_token:.3f} → 1文字あたり {per_char:.3f}")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
1トークンあたり 4.976 → 1文字あたり 3.225
```

</details>

1文字あたりに直すと **約3.23**。第12章の文字版（同じ大きさのモデル・同じステップ数で **3.10**）と比べると、BPE版のほうが**少し悪い**、という結果になりました。

**ミナト**：あれ？ 本物のLLMはみんなBPEなのに、文字版に負けちゃうんですか？

**ボンゴ先生**：私もそう予想していた（白状すると、この章を書く前はそう信じていた）。でも、実験の結果はこれだ。予想が外れたときこそ、考えるチャンスだよ。学習の出力をよく見てごらん。1000ステップから1500ステップで、検証の損失が4.930から4.976に**悪化**している。過学習が始まっているんだ。

> **💡 理論のポイント：BPEの利点が出るのは、データが多いとき**
>
> BPEにすると、語彙が2061文字から2000トークンに変わるだけでなく、**1つ1つのトークンが出てくる回数が減ります**。「話を」というトークンは、「話」という文字よりずっと出てくる回数が少ないですよね。『こころ』の14万字では、それぞれのトークンをしっかり学ぶには例が足りず、丸暗記（過学習）しやすくなるのです。
>
> 一方、データが何億・何兆トークンもあれば、どのトークンも十分に学べるので、「同じ窓で遠くまで見渡せる」「言い回しを1つの部品として扱える」というBPEの利点が勝ちます。本物のLLMがみなBPEのようなトークナイザーを使うのは、そのためです。**手法の良し悪しは、データの大きさとセットで考える**。これも大事な教訓です。

> **⚠️ つまずき注意：比較の注意**
>
> ただし、この比較には少しだけ「ずる」があります。BPEのトークナイザーは『こころ』**全体**（検証データの部分も含む）で学習させたので、検証データによく出る言い回しを知っています。厳密な実験では、トークナイザーも訓練データだけで作るべきです。研究で数字を比べるときは、こういう**データの漏れ**（leakage）に気をつけましょう（第20章で詳しく）。

<details><summary><b>もっと深く：日本語のトークナイズと、ほかの方式</b>（クリックで開く）</summary>

- **事前分割**：GPT-2などは、BPEの前に正規表現で「単語・数字・記号」に分けておき、単語をまたぐ合体を禁止しています（第2章の `' am'` のように空白が単語の前にくっつくのはこのため）。日本語は単語の間に空白がないので、この分割があまり効きません。
- **形態素解析**：日本語では昔から、MeCabなどの辞書を使って単語（形態素）に分ける方法がよく使われてきました。日本語特化のLLMの中には、形態素解析とBPEを組み合わせるものもあります。
- **SentencePiece / Unigram**：BPEとは逆に、大きな語彙から始めて「消しても困らないトークン」を削っていく方式（Unigram言語モデル）もあります。Googleの `SentencePiece` というライブラリが有名で、多くの多言語モデルが使っています。
- **Unicode正規化**：「Ａ」（全角）と「A」（半角）、「ｶ」と「カ」のように、見た目が同じでも文字コードが違う文字があります。`unicodedata.normalize("NFKC", text)` で統一してからトークナイズすると、語彙の無駄が減ります。

</details>

#### 📝 確認問題（`ch14-q12` · Lv.3 · 異なるトークナイザー間の公正な比較：1文字あたりの損失への換算）

あるBPEモデルの検証データに対する「1トークンあたりの損失」が **3.0** でした。この検証データは **10,000文字** で、BPEでエンコードすると **4,000トークン** になります。文字単位モデルと公正に比べるための「**1文字あたりの損失**」はいくらですか？（小数で答えてください）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：トークナイザーが違うモデルを公平に比べるには、まず文章全体の総損失を求め、共通の単位（文字数）で割り直します。
- **ヒント 2**：文章全体の総損失は「1トークンあたりの損失 $3.0 \times 4{,}000$ トークン」で求まります。それを文字数 $10{,}000$ で割りましょう。
- **正解**：`1.2`

**考え方**：トークナイザーが異なると「1トークンに含まれる文字数」が違うため、そのまま1トークン損失を比べるのはメートルとインチを比べるようなものです。文章全体の損失を文字数（またはバイト数）で割って単位をそろえます。

1. 文章全体の総損失 $= 3.0 \times 4{,}000 = 12{,}000$。
1. これを文字数 $10{,}000$ で割る：$12{,}000 \div 10{,}000 = 1.2$。
1. したがって1文字あたりの損失は `1.2`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html#ex-ch14-q12

</details>

#### 📝 確認問題（`ch14-q13` · Lv.4 · 少データでのBPEの過学習とデータ漏れ（leakage）の注意点）

『こころ』1冊（約14万字）の実験では、1文字あたり損失でBPE版（3.23）が文字版（3.10）にわずかに負けました。①なぜデータが少ないとBPE版が過学習しやすくなるのか、②厳密な実験でトークナイザーを学習するときに気をつけるべき「データの漏れ（leakage）」とは何か、の2点を説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：①1トークンあたりの出現回数と過学習の関係、②トークナイザー学習時に使ってよいデータの範囲、の2軸で整理しましょう。
- **ヒント 2**：①長いサブワードはデータが少ないと出現回数が分散して丸暗記しやすい点、②検証データをBPE学習に混ぜると未知データ特有の表現が語彙に漏れる点を具体例と合わせてまとめましょう。
- **解答の目安**：①BPEでは複数の文字をまとめたトークン（例：「話を」「ありません」）が増えるため、文字単位に比べて1つ1つのトークンが訓練データ中に現れる回数が減り、少ない用例から丸暗記（過学習）しやすくなります（大量データなら十分学べるためBPEが優位になります）。②データの漏れとは、検証データも含めた全テキストでトークナイザーを学習してしまうことで、検証データ特有の言い回しが事前に語彙へ紛れ込む現象です。厳密な評価ではトークナイザーも訓練データだけで学習する必要があります。

**考え方**：手法の優劣はデータ規模と切り離せないこと、また前処理（トークナイザー構築）段階でも訓練と検証を厳密に分離することが、LLM研究の基本姿勢です。

1. ① 出現頻度の分散：同じ文章量でも「話」より「話を」の出現回数は少なく、表現を汎化しにくい。
1. ② データ漏れ防止：トークナイザーの `train` に検証データを含めると、未知データ評価の公正さが損なわれる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html#ex-ch14-q13

</details>

**✅ できたらチェック**

- [ ] BPEのデモで、合体をくり返すと列が短くなるのを見た
- [ ] `get_stats` と `merge` を自分で書いて動かした
- [ ] 300回の合体で、「ありません」「私には」のようなトークンが生まれるのを見た
- [ ] 自作の `encode` と `decode` で、文章を往復できた
- [ ] `tokenizers` で高速にBPEを学習させた
- [ ] BPE版GPTの損失を「1文字あたり」に直して、文字版と比べた

> **📝 まとめ**
>
> - **BPE**：バイトから始めて、いちばん多い隣どうしのペアを合体させる、をくり返す
> - 合体ルールは**学習した順番**でエンコードに使う。デコードはバイトをつなげるだけ
> - 語彙サイズは、列の短さとパラメータ数のトレードオフ
> - トークナイザーが違うモデルは、**1文字（1バイト）あたりの損失**に直して比べる
> - データの**漏れ**に注意。トークナイザーも本当は訓練データだけで作る

**ボンゴ先生**：トークナイザーまで自作できた。これでキミのGPTは、部品が全部そろったことになる。次の章では、とっておきの実験をする。OpenAIが公開した**本物のGPT-2の重み**を、キミが書いたGPTのコードに読み込むんだ。キミのコードが、本物と同じ答えを出すかどうか。答え合わせの時間だ。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/14-bpe.html
- **次の章**：[第15章 自分のコードに本物のGPT-2を入れる](https://sakunokyumei.github.io/llm-research-textbook/15-gpt2.html)